# Detector de ladridos para webcam-bot

Este notebook entrena el modelo que usará el bot para saber si un perro está
ladrando, a partir del sonido del micrófono.

**Cómo funciona:**

1. Usa **YAMNet**, un modelo de Google ya entrenado que reconoce 521 tipos de
   sonido (entre ellos, ladridos). YAMNet convierte cada trozo de ~1 segundo de
   audio en un «resumen» de 1024 números (un *embedding*).
2. Sobre esos resúmenes entrena una **red pequeña** que decide solo una cosa:
   ¿ladrido o no ladrido?
3. Exporta YAMNet + la red pequeña como **un único archivo `ladridos.tflite`**
   que recibe el audio tal cual y devuelve la probabilidad de ladrido.

**Datos:** los ladridos y demás sonidos de la base de datos pública
[ESC-50](https://github.com/karolpiczak/ESC-50) y, si las tienes, **tus propias
grabaciones** (ver sección 3). Cuantas más grabaciones propias, mejor
funcionará con tu perro, tu micrófono y el ruido de tu casa.

**Resultado:** dos archivos que hay que copiar a la carpeta `models/` del bot:

| Archivo | Contenido |
|---|---|
| `ladridos.tflite` | El modelo |
| `ladridos_info.json` | Formato de entrada, umbral recomendado y métricas |

**Cómo ejecutarlo en Colab:** menú *Entorno de ejecución → Ejecutar todas*.
No hace falta GPU: con la CPU normal tarda unos 10–15 minutos, casi todo en
descargar ESC-50 y procesar los audios.

## 1. Librerías y configuración

Todas estas librerías ya vienen instaladas en Colab.

In [ ]:
import datetime
import json
import os
import tarfile
import urllib.request
import zipfile

import IPython.display
import keras
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import confusion_matrix, roc_auc_score

keras.utils.set_random_seed(42)  # para que los resultados sean repetibles
print("TensorFlow", tf.__version__)

In [ ]:
# --- Formato de audio (NO cambiar: es el que espera YAMNet) ---
SAMPLE_RATE = 16000     # muestras por segundo, mono
WINDOW_SAMPLES = 15600  # 0,975 s: un trozo de audio = una predicción
HOP_SAMPLES = 7680      # 0,48 s: YAMNet analiza trozos solapados cada 0,48 s

# --- Fuentes ---
# YAMNet se descarga directamente de Kaggle (su hogar actual, antes TF Hub)
YAMNET_URL = "https://www.kaggle.com/api/v1/models/google/yamnet/tensorFlow2/yamnet/1/download"
YAMNET_DIR = "yamnet"
ESC50_URL = "https://github.com/karolpiczak/ESC-50/archive/master.zip"
ESC50_DIR = "ESC-50-master"
MIS_GRABACIONES_DIR = "mis_grabaciones"

# En los clips de ladridos hay silencios entre ladrido y ladrido. Los trozos
# con menos volumen que este porcentaje del trozo más fuerte del clip se
# descartan, para no enseñar al modelo que «silencio = ladrido».
MIN_RMS_RELATIVO = 0.25

# --- Salida ---
SALIDA_TFLITE = "ladridos.tflite"
SALIDA_INFO = "ladridos_info.json"

## 2. Descargar ESC-50

ESC-50 tiene 2000 clips de 5 segundos de 50 tipos de sonido (40 de cada uno):
ladridos, maullidos, lluvia, voces, puertas, aspiradora… Los 40 clips de
ladridos (`dog`) son los ejemplos de «ladrido»; los otros 1960, de «no
ladrido».

ESC-50 viene dividida en 5 partes (*folds*). Se usan así:

- Partes 1–3 → **entrenamiento**: con lo que aprende el modelo.
- Parte 4 → **validación**: para decidir cuándo parar de entrenar y elegir el
  umbral.
- Parte 5 → **prueba**: audios que el modelo nunca ha visto, para medir cómo
  funciona de verdad.

In [ ]:
if not os.path.isdir(ESC50_DIR):
    print("Descargando ESC-50 (~600 MB)...")
    urllib.request.urlretrieve(ESC50_URL, "esc50.zip")
    with zipfile.ZipFile("esc50.zip") as z:
        z.extractall(".")
    os.remove("esc50.zip")

esc50 = pd.read_csv(f"{ESC50_DIR}/meta/esc50.csv")
esc50 = pd.DataFrame({
    "path": ESC50_DIR + "/audio/" + esc50["filename"],
    "label": (esc50["category"] == "dog").astype(int),
    "categoria": esc50["category"],
    "split": esc50["fold"].map({1: "train", 2: "train", 3: "train", 4: "val", 5: "test"}),
    "origen": "ESC-50",
})
print(f"{len(esc50)} clips, {esc50['label'].sum()} de ladridos")

## 3. Tus propias grabaciones (opcional)

Si aún no tienes grabaciones, **sáltate esta sección**: el notebook funciona
solo con ESC-50.

Cuando las tengas, organízalas así:

```
mis_grabaciones/
├── ladrido/        ← tu perro ladrando (cuanto más variado, mejor)
└── no_ladrido/     ← ruido normal de casa: tele, voces, puertas, silencio...
```

- Formatos: `.wav`, `.mp3`, `.ogg`, `.flac` o `.m4a`. Da igual la duración.
- Si en un archivo de `ladrido/` hay silencios entre ladridos, no pasa nada:
  el notebook descarta automáticamente los trozos de poco volumen.
- **Lo más útil**: grabarlo con el mismo micrófono y en el mismo sitio donde
  va a estar instalado. Y grabar también bastante `no_ladrido`: es lo que
  evita falsos avisos.

Para subirlas a Colab: comprime la carpeta como `mis_grabaciones.zip` y
arrástrala al panel de archivos (icono de carpeta, a la izquierda). La celda
siguiente la descomprime.

Tus grabaciones se reparten al azar: 70 % entrenamiento, 15 % validación y
15 % prueba.

In [ ]:
EXTENSIONES = (".wav", ".mp3", ".ogg", ".flac", ".m4a")

zip_grabaciones = MIS_GRABACIONES_DIR + ".zip"
if os.path.isfile(zip_grabaciones) and not os.path.isdir(MIS_GRABACIONES_DIR):
    with zipfile.ZipFile(zip_grabaciones) as z:
        z.extractall(MIS_GRABACIONES_DIR)
# Si el zip contenía la carpeta mis_grabaciones/ dentro, se usa esa
base = MIS_GRABACIONES_DIR
if os.path.isdir(os.path.join(base, MIS_GRABACIONES_DIR)):
    base = os.path.join(base, MIS_GRABACIONES_DIR)

filas = []
for carpeta, label in (("ladrido", 1), ("no_ladrido", 0)):
    directorio = os.path.join(base, carpeta)
    if not os.path.isdir(directorio):
        continue
    for nombre in sorted(os.listdir(directorio)):
        if nombre.lower().endswith(EXTENSIONES):
            filas.append({
                "path": os.path.join(directorio, nombre),
                "label": label,
                "categoria": f"propia_{carpeta}",
                "origen": "propias",
            })

propias = pd.DataFrame(filas, columns=["path", "label", "categoria", "origen"])
if len(propias):
    rng = np.random.default_rng(42)
    propias["split"] = rng.choice(["train", "val", "test"], size=len(propias), p=[0.7, 0.15, 0.15])
    print(f"Grabaciones propias: {len(propias)} "
          f"({propias['label'].sum()} de ladrido, {(propias['label'] == 0).sum()} sin ladrido)")
else:
    print("No hay grabaciones propias: se entrena solo con ESC-50.")

datos = pd.concat([esc50, propias], ignore_index=True)
datos["label"] = datos["label"].astype(int)
pd.crosstab(datos["origen"] + " / " + datos["label"].map({1: "ladrido", 0: "no ladrido"}),
            datos["split"])

## 4. Cargar YAMNet

Antes de entrenar nada, una prueba: YAMNet con un clip de ladridos. Debería
reconocer «Dog», «Bark», «Bow-wow»… entre los sonidos más probables.

In [ ]:
if not os.path.isfile(os.path.join(YAMNET_DIR, "saved_model.pb")):
    urllib.request.urlretrieve(YAMNET_URL, "yamnet.tar.gz")
    with tarfile.open("yamnet.tar.gz") as t:
        t.extractall(YAMNET_DIR, filter="data")
    os.remove("yamnet.tar.gz")

yamnet = tf.saved_model.load(YAMNET_DIR)
clases_yamnet = pd.read_csv(yamnet.class_map_path().numpy().decode())["display_name"].tolist()


def cargar_audio(path):
    """Carga un audio como 16 kHz mono float32 en [-1, 1], el formato de YAMNet."""
    wav, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    return wav.astype(np.float32)


ejemplo = datos[(datos["origen"] == "ESC-50") & (datos["label"] == 1)].iloc[0]["path"]
wav = cargar_audio(ejemplo)
scores, _, _ = yamnet(wav)
# Puntuación máxima de cada sonido a lo largo del clip (hay silencios entre ladridos)
maximo = scores.numpy().max(axis=0)
for i in maximo.argsort()[::-1][:5]:
    print(f"{clases_yamnet[i]:<25} {maximo[i]:.2f}")
IPython.display.Audio(wav, rate=SAMPLE_RATE)

## 5. Convertir todos los audios en embeddings

Cada clip se trocea en ventanas de 0,975 s (solapadas cada 0,48 s) y YAMNet
convierte cada ventana en un embedding de 1024 números. Es lo más lento del
notebook (unos minutos).

En los clips de ladrido se descartan las ventanas de poco volumen (los
silencios entre ladridos), según `MIN_RMS_RELATIVO`.

In [ ]:
def rms_por_ventana(wav, n_ventanas):
    """Volumen (RMS) de cada ventana, con el mismo troceado que usa YAMNet."""
    rms = np.zeros(n_ventanas, dtype=np.float32)
    for i in range(n_ventanas):
        trozo = wav[i * HOP_SAMPLES: i * HOP_SAMPLES + WINDOW_SAMPLES]
        if len(trozo):
            rms[i] = np.sqrt(np.mean(trozo ** 2))
    return rms


X, y, split, clip_id = [], [], [], []
for n, (idx, fila) in enumerate(datos.iterrows(), start=1):
    wav = cargar_audio(fila["path"])
    if len(wav) < WINDOW_SAMPLES:
        wav = np.pad(wav, (0, WINDOW_SAMPLES - len(wav)))
    _, embeddings, _ = yamnet(wav)
    embeddings = embeddings.numpy()

    rms = rms_por_ventana(wav, len(embeddings))
    if fila["label"] == 1:
        conservar = rms >= MIN_RMS_RELATIVO * rms.max()
    else:
        conservar = np.ones(len(embeddings), dtype=bool)

    X.append(embeddings[conservar])
    y += [fila["label"]] * int(conservar.sum())
    split += [fila["split"]] * int(conservar.sum())
    clip_id += [idx] * int(conservar.sum())

    if n % 250 == 0 or n == len(datos):
        print(f"{n}/{len(datos)} audios procesados")

X = np.concatenate(X)
y = np.array(y, dtype=int)
split = np.array(split)
clip_id = np.array(clip_id)
entrenamiento, validacion, prueba = (split == "train"), (split == "val"), (split == "test")

print(f"\n{len(X)} ventanas en total")
for nombre, m in (("entrenamiento", entrenamiento), ("validación", validacion), ("prueba", prueba)):
    print(f"  {nombre:<14} {m.sum():>6} ventanas, {y[m].sum():>4} de ladrido")

## 6. Entrenar el clasificador

Una red pequeña: 1024 números de entrada → 256 neuronas → 1 salida (la
probabilidad de ladrido).

Hay muchísimos más ejemplos de «no ladrido» que de «ladrido», así que cada
ejemplo de ladrido pesa más en el entrenamiento (`class_weight`). Si no, el
modelo aprendería a responder siempre «no ladrido» y acertaría el 98 % de las
veces sin servir para nada.

El entrenamiento se para solo cuando deja de mejorar en validación.

In [ ]:
n_pos = int(y[entrenamiento].sum())
n_neg = int((y[entrenamiento] == 0).sum())
class_weight = {0: 1.0, 1: n_neg / n_pos}

clasificador = keras.Sequential([
    keras.Input(shape=(1024,)),
    keras.layers.Dense(256, activation="relu", name="oculta"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(1, activation="sigmoid", name="salida"),
])
clasificador.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=[keras.metrics.AUC(name="auc")],
)
historial = clasificador.fit(
    X[entrenamiento], y[entrenamiento],
    validation_data=(X[validacion], y[validacion]),
    epochs=100,
    batch_size=64,
    class_weight=class_weight,
    callbacks=[keras.callbacks.EarlyStopping(
        monitor="val_auc", mode="max", patience=10, restore_best_weights=True)],
    verbose=2,
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(historial.history["loss"], label="entrenamiento")
ax1.plot(historial.history["val_loss"], label="validación")
ax1.set_title("Pérdida (más bajo = mejor)")
ax1.legend()
ax2.plot(historial.history["auc"], label="entrenamiento")
ax2.plot(historial.history["val_auc"], label="validación")
ax2.set_title("AUC (más alto = mejor, máximo 1)")
ax2.legend()
plt.show()

## 7. Elegir el umbral y evaluar

El modelo devuelve una probabilidad entre 0 y 1; el **umbral** es a partir de
qué valor se considera ladrido.

Se usa **0,5**. En el bot, un falso aviso no es grave: antes de reproducir
el audio, la cámara comprueba que haya un perro. Lo que sí importa es no
perder ladridos.

No conviene subirlo mucho aunque la tabla de abajo lo haga parecer gratis:
con los audios de ESC-50 el modelo está muy seguro (casi todo sale cerca de
0 o de 1), pero con tu micrófono el sonido será distinto y las
probabilidades menos extremas. Un umbral muy alto (0,99) perdería ladridos
reales.

La tabla muestra qué pasaría con otros umbrales en validación. Si con tu
micrófono no detecta bien o da demasiados avisos, puedes cambiar el umbral en
`ladridos_info.json` sin reentrenar.

Después se mide con los audios de **prueba**, que el modelo no ha visto
nunca, de dos formas:

- **Por ventana** (cada trozo de ~1 s).
- **Por clip**: un clip cuenta como ladrido si alguna de sus ventanas supera
  el umbral. Es lo más parecido a cómo lo usará el bot.

In [ ]:
UMBRAL = 0.5

p_val = clasificador.predict(X[validacion], verbose=0).ravel()

# Qué pasaría con otros umbrales (en validación)
tabla = []
for u in (0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9, 0.99):
    pred = p_val >= u
    tabla.append({
        "umbral": u,
        "ladridos detectados": f"{(pred & (y[validacion] == 1)).sum()}/{(y[validacion] == 1).sum()}",
        "falsos avisos": f"{(pred & (y[validacion] == 0)).sum()}/{(y[validacion] == 0).sum()}",
    })
display(pd.DataFrame(tabla).set_index("umbral"))

p_test = clasificador.predict(X[prueba], verbose=0).ravel()


def metricas(reales, predichos):
    tn, fp, fn, tp = confusion_matrix(reales, predichos, labels=[0, 1]).ravel()
    return {
        "ladridos_detectados": f"{tp}/{tp + fn}",
        "recall": round(tp / max(tp + fn, 1), 3),
        "precision": round(tp / max(tp + fp, 1), 3),
        "falsos_avisos": f"{fp}/{fp + tn}",
    }


# Por ventana
auc_test = roc_auc_score(y[prueba], p_test)
metricas_ventana = metricas(y[prueba], p_test >= UMBRAL)

# Por clip: la probabilidad del clip es la máxima de sus ventanas
por_clip = pd.DataFrame({"clip": clip_id[prueba], "p": p_test}).groupby("clip")["p"].max()
clips = datos.loc[por_clip.index].assign(p=por_clip.values)
metricas_clip = metricas(clips["label"], clips["p"] >= UMBRAL)

print(f"AUC en prueba: {auc_test:.3f}")
display(pd.DataFrame({"Por ventana": metricas_ventana, "Por clip": metricas_clip}))

falsos = clips[(clips["label"] == 0) & (clips["p"] >= UMBRAL)]
if len(falsos):
    print("Sonidos que el modelo confunde con ladridos (clips de prueba):")
    display(falsos["categoria"].value_counts().to_frame("clips"))

**Cómo leerlo:**

- **recall**: de los ladridos reales, cuántos detecta (1 = todos).
- **precision**: de lo que dice que es ladrido, cuánto lo es de verdad
  (1 = ningún falso aviso).
- Con solo 8 clips de ladridos en prueba, cada clip son 12,5 puntos de
  *recall*: los números bailan bastante. Con grabaciones propias las medidas
  serán más fiables.

## 8. Exportar a TensorFlow Lite

Se junta YAMNet + el clasificador en un único modelo:

- **Entrada**: `audio`, 15600 números float32 (0,975 s a 16 kHz, mono, entre
  -1 y 1).
- **Salida**: `probabilidad_ladrido`, un número entre 0 y 1.

Se exporta en formato **TensorFlow Lite** (`.tflite`) usando solo
operaciones estándar, para que el bot pueda ejecutarlo con la librería
ligera `ai-edge-litert` en vez de TensorFlow completo.

In [ ]:
class DetectorLadridos(tf.Module):
    def __init__(self, yamnet, clasificador):
        super().__init__()
        self.yamnet = yamnet
        oculta = clasificador.get_layer("oculta").get_weights()
        salida = clasificador.get_layer("salida").get_weights()
        self.w1 = tf.constant(oculta[0])
        self.b1 = tf.constant(oculta[1])
        self.w2 = tf.constant(salida[0])
        self.b2 = tf.constant(salida[1])

    @tf.function(input_signature=[tf.TensorSpec([WINDOW_SAMPLES], tf.float32, name="audio")])
    def __call__(self, audio):
        _, embeddings, _ = self.yamnet(audio)
        oculta = tf.nn.relu(tf.matmul(embeddings, self.w1) + self.b1)
        probabilidad = tf.sigmoid(tf.matmul(oculta, self.w2) + self.b2)
        return {"probabilidad_ladrido": tf.reshape(probabilidad, [1])}


detector = DetectorLadridos(yamnet, clasificador)
tf.saved_model.save(detector, "detector_ladridos",
                    signatures=detector.__call__.get_concrete_function())

converter = tf.lite.TFLiteConverter.from_saved_model("detector_ladridos")
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]
modelo_tflite = converter.convert()
with open(SALIDA_TFLITE, "wb") as f:
    f.write(modelo_tflite)
print(f"{SALIDA_TFLITE}: {len(modelo_tflite) / 1e6:.1f} MB")

## 9. Comprobar el modelo exportado

Se compara el `.tflite` con el modelo original en ventanas de audio de
prueba: tienen que dar la misma probabilidad. Si esta celda falla, **no
uses el `.tflite`**.

In [ ]:
interprete = tf.lite.Interpreter(model_content=modelo_tflite)
ejecutar_tflite = interprete.get_signature_runner()

# Una ventana (la de más volumen) de cada clip de prueba de ladrido y de
# 20 clips de prueba sin ladrido
clips_prueba = pd.concat([
    datos[(datos["split"] == "test") & (datos["label"] == 1)],
    datos[(datos["split"] == "test") & (datos["label"] == 0)].sample(20, random_state=42),
])
diferencias = []
for _, fila in clips_prueba.iterrows():
    wav = cargar_audio(fila["path"])
    if len(wav) < WINDOW_SAMPLES:
        wav = np.pad(wav, (0, WINDOW_SAMPLES - len(wav)))
    rms = rms_por_ventana(wav, (len(wav) - WINDOW_SAMPLES) // HOP_SAMPLES + 1)
    inicio = int(rms.argmax()) * HOP_SAMPLES
    ventana = wav[inicio: inicio + WINDOW_SAMPLES]

    _, embeddings, _ = yamnet(ventana)
    p_original = float(clasificador.predict(embeddings.numpy(), verbose=0)[0, 0])
    p_tflite = float(ejecutar_tflite(audio=ventana)["probabilidad_ladrido"][0])
    diferencias.append(abs(p_original - p_tflite))

print(f"Diferencia máxima entre el modelo original y el .tflite: {max(diferencias):.6f}")
assert max(diferencias) < 1e-3, "El .tflite no da los mismos resultados: no lo uses"
print("✅ El .tflite funciona igual que el modelo original")

## 10. Guardar y descargar

Se guarda `ladridos_info.json` con todo lo que necesita saber el bot, y se
descargan los dos archivos.

In [ ]:
info = {
    "modelo": SALIDA_TFLITE,
    "creado": datetime.datetime.now().isoformat(timespec="seconds"),
    "entrada": {
        "nombre": "audio",
        "sample_rate": SAMPLE_RATE,
        "muestras": WINDOW_SAMPLES,
        "formato": "float32 mono en [-1, 1]",
    },
    "salida": {"nombre": "probabilidad_ladrido", "rango": [0, 1]},
    "umbral_recomendado": round(UMBRAL, 3),
    "metricas_prueba": {
        "auc_ventana": round(float(auc_test), 3),
        "por_ventana": metricas_ventana,
        "por_clip": metricas_clip,
    },
    "datos": {
        "clips_esc50": int((datos["origen"] == "ESC-50").sum()),
        "grabaciones_propias": int((datos["origen"] == "propias").sum()),
    },
}
with open(SALIDA_INFO, "w", encoding="utf-8") as f:
    json.dump(info, f, ensure_ascii=False, indent=2)
print(json.dumps(info, ensure_ascii=False, indent=2))

try:
    from google.colab import files
    files.download(SALIDA_TFLITE)
    files.download(SALIDA_INFO)
except ImportError:
    print(f"\nNo estás en Colab: los archivos están en {os.path.abspath('.')}")

## Siguientes pasos

1. Copia `ladridos.tflite` y `ladridos_info.json` a la carpeta `models/` del
   bot (junto a los archivos del detector de perro).
2. Cuando tengas grabaciones de tu perro, ponlas en `mis_grabaciones/` (ver
   sección 3), vuelve a ejecutar el notebook y sustituye los dos archivos.